<a href="https://colab.research.google.com/github/AlirezPa/MLZNU03/blob/main/IRWS_HM01-1.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Download and extract

In [ ]:
# مرحله ۱: نصب همه پکیج‌های لازم (یک بار برای همیشه)
!pip install -q whoosh

# مرحله ۲: ری‌استارت خودکار ران‌تایم (مهم!)
import os
import IPython
IPython.display.clear_output(wait=False)
print("نصب تموم شد، الان ران‌تایم ری‌استارت می‌شه...")
os.kill(os.getpid(), 9)

In [1]:
import os
import shutil
from whoosh.index import create_in, open_dir
from whoosh.fields import Schema, TEXT, ID
from whoosh.analysis import RegexTokenizer, LowercaseFilter
from whoosh.qparser import MultifieldParser, QueryParser
from whoosh import scoring
import itertools

In [2]:
!gdown 1N5V40HQsnCHHytdmG98YfXvN_3DCjV67

Downloading...
From (original): https://drive.google.com/uc?id=1N5V40HQsnCHHytdmG98YfXvN_3DCjV67
From (redirected): https://drive.google.com/uc?id=1N5V40HQsnCHHytdmG98YfXvN_3DCjV67&confirm=t&uuid=665f9aa3-5b4d-4b9a-925a-0bfbb5bd5990
To: /content/dataframe_hamshahri2.tar
100% 160M/160M [00:04<00:00, 39.3MB/s]


In [3]:
!tar -xf dataframe_hamshahri2.tar

# Load data

In [4]:
import pandas as pd

df_docs=pd.read_csv('/content/hamshahri2/docs.csv.gz', compression='gzip')
df_judgments=pd.read_csv('/content/hamshahri2/judgments_dataframe.csv.gz', compression='gzip')
df_queries_fa=pd.read_csv('/content/hamshahri2/queries_fa.csv.gz', compression='gzip')

In [5]:
from whoosh.analysis import RegexTokenizer, LowercaseFilter

# ساخت analyzer ساده (جایگزین WhitespaceAnalyzer)
simple_analyzer = RegexTokenizer() | LowercaseFilter()

def build_index(index_name, content_mode="both"):
    schema = Schema(doc_id=ID(stored=True),
                    content=TEXT(analyzer=simple_analyzer))  # اینجا تغییر کرد

    index_path = f"index_{index_name}"
    if os.path.exists(index_path):
        import shutil
        shutil.rmtree(index_path)

    os.mkdir(index_path)
    ix = create_in(index_path, schema)
    writer = ix.writer()

    for doc_id, doc in enumerate(docs):
        if content_mode == "title":
            text_to_index = doc.get("title", "")
        elif content_mode == "text":
            text_to_index = doc.get("text", "")
        elif content_mode == "both":
            text_to_index = doc.get("title", "") + " " + doc.get("text", "")
        else:
            raise ValueError("mode اشتباه!")

        writer.add_document(doc_id=str(doc_id),
                            content=text_to_index)

    writer.commit()
    print(f"ایندکس {index_name} با موفقیت ساخته شد ({content_mode})")
    return index_path

In [12]:
qrels={}
for item in df_judgments[df_judgments['relevancy'] == 1].groupby('query_id')[['doc_id', 'relevancy']]:
  qrels[str(item[0])]={val[0]:int(val[1]) for val in zip(item[1]['doc_id'],item[1]['relevancy'])}

In [7]:
df_docs

,Unnamed: 0,DOCID,TITLE,CAT_FA,DATE_FA,TEXT,ORIGINALFILE,ISSUE,DATE_EN,CAT_EN
0,0,HAM2-851017-001,\nطي مراسمي با حضور انديشمندان ايراني و خارجي ...,ادب و هنر,1385/10/17,\n,/1385/851017/news/_adabh.htm,يكشنبه 17 دي 1385 - سال چهاردهم - شماره4177 - ...,2007-01-07,Literature and Art
1,1,HAM2-851017-002,\nباحضور استادان و كارشناسان خارجي برگزار مي ش...,ادب و هنر,1385/10/17,\n,/1385/851017/news/_adabh.htm,يكشنبه 17 دي 1385 - سال چهاردهم - شماره4177 - ...,2007-01-07,Literature and Art
2,2,HAM2-851017-003,\nجشنواره فيلم هاي ايراني در لاهور پاكستان\n,ادب و هنر,1385/10/17,\n,/1385/851017/news/_adabh.htm,يكشنبه 17 دي 1385 - سال چهاردهم - شماره4177 - ...,2007-01-07,Literature and Art
3,3,HAM2-851017-004,\nبا رهبري مجيد انتظامي در اصفهان\nسمفوني ايثا...,ادب و هنر,1385/10/17,\n,/1385/851017/news/_adabh.htm,يكشنبه 17 دي 1385 - سال چهاردهم - شماره4177 - ...,2007-01-07,Literature and Art
4,4,HAM2-851017-005,\nبر ديوار نگار خانه ها\n,ادب و هنر,1385/10/17,\nگالري آريا: نمايشگاه نقاشي فريما فرهت نيا و ...,/1385/851017/news/_adabh.htm,يكشنبه 17 دي 1385 - سال چهاردهم - شماره4177 - ...,2007-01-07,Literature and Art
...,...,...,...,...,...,...,...,...,...,...
318619,318619,HAM2-840516-156,\nاخبار دانشبراي اولين بار در كشور\nسيستم لايه...,علمی فرهنگی.علمی,1384/05/16,\nسيستم لايه نشاني dc rf اسپاترينگ، براي اولين...,/1384/840516/world/_sciew.htm,يكشنبه 16 مرداد 1384,2005-08-07,Science and Culture.Science
318620,318620,HAM2-840516-157,\nتنفيذ رياست جمهوري\n,سیاسی,1384/05/16,\n,/1384/840516/world/_siasatw.htm,يكشنبه 16 مرداد 1384,2005-08-07,Politics
318621,318621,HAM2-840516-158,\nبه انگيزه سالروز ميلاد مبارك امام محمد باقر(...,سیاسی,1384/05/16,\n,/1384/840516/world/_siasatw.htm,يكشنبه 16 مرداد 1384,2005-08-07,Politics
318622,318622,HAM2-840516-159,\nگفت وگو با مربي جديد تيم نوجوانان ايران\nتوص...,ورزش,1384/05/16,\n,/1384/840516/world/_sporw.htm,يكشنبه 16 مرداد 1384,2005-08-07,Sport


In [ ]:
df_judgments

,Unnamed: 0,query_id,doc_id,relevancy
0,0,1,HAM2-751213-029,0.0
1,1,1,HAM2-751216-038,0.0
2,2,1,HAM2-751223-037,0.0
3,3,1,HAM2-760131-065,0.0
4,4,1,HAM2-760222-025,0.0
...,...,...,...,...
21767,21767,50,HAM2-850908-069,0.0
21768,21768,50,HAM2-850912-027,0.0
21769,21769,50,HAM2-851017-082,0.0
21770,21770,50,HAM2-851117-091,0.0


In [ ]:
df_queries_fa

,Unnamed: 0,ID,TITLE,DESCRIPTION,NARRATIVE
0,0,1,بازسازي شهر زلزله زده بم,بازسازي بناها ي شهر بم و ويرانه هاي اطراف آن,اقدامات و كمك هاي مردمي و دولتي به منظور كمك ر...
1,1,2,برگزيدگان جشنواره فيلم فجر,فيلم هاي برگزيده در جشنواره فيلم فجر,به دنبال فهرست و اسامي فيلم هاي برگزيده در جشن...
2,2,3,علل مرگ و مير نهنگ ها,علل و عوامل مختلف مرگ و مير نهنگ ها در جهان,به دنبال علل و عوامل مرگ نهنگ ها نظير صيد و شك...
3,3,4,راه هاي جلو گيري از ابتلا به ديابت\n,علايم وجود بيماري ديابت و راه هاي پيشگيري از ا...,به دنبال اطلاعات مربوط به پيشگيري و جلوگيري از...
4,4,5,انتخاب سرمربي تيم ملي فوتبال ايران,اظهار نظر فدراسيون درمورد انتخاب سرمربي جديد ت...,به دنبال اطلاعات مربوط به گزينه هاي فدراسيون ب...
5,5,6,آمار مبتلايان به ايدز در ايران,آمار مبتلايان به ايدز و ميانگين سني آن ها در ا...,اطلاعات ذخيره شده در مورد مبتلايان به بيماري ا...
6,6,7,عرضه سهام شركت ايران خودرو,خريد و فروش سهام شركت ايران خودرو در بازار بورس,اطلاعات موجود در مورد قيمت و زمان فروش سهام شر...
7,7,8,نفرات برگزيده كنكور سراسري,نفرات برگزيده كنكور سراسري در تمام زير گروه ها...,اطلاعات سازمان سنجش آموزش كشور در مورد نفرات ب...
8,8,9,جبران خسارات ناشي از خشكسالي,اقدامات دولت در راستاي جبران خسارات ناشي از خش...,به دنبال اقدامات دولت در راستاي جبران ضرر و زي...
9,9,10,بهره برداري از تونل رسالت,بهره برداري و افتتاح تونل شهري رسالت تهران,به دنبال اطلاعات مربوط به تاريخ بهره برداري و ...


In [17]:
qrels

{'1': {'HAM2-821006-059': 1,
  'HAM2-821006-060': 1,
  'HAM2-821007-088': 1,
  'HAM2-821007-097': 1,
  'HAM2-821009-038': 1,
  'HAM2-821009-048': 1,
  'HAM2-821009-067': 1,
  'HAM2-821009-090': 1,
  'HAM2-821009-109': 1,
  'HAM2-821009-115': 1,
  'HAM2-821009-121': 1,
  'HAM2-821010-030': 1,
  'HAM2-821010-076': 1,
  'HAM2-821010-109': 1,
  'HAM2-821010-114': 1,
  'HAM2-821011-026': 1,
  'HAM2-821011-049': 1,
  'HAM2-821011-071': 1,
  'HAM2-821011-073': 1,
  'HAM2-821011-081': 1,
  'HAM2-821011-127': 1,
  'HAM2-821013-073': 1,
  'HAM2-821014-059': 1,
  'HAM2-821014-070': 1,
  'HAM2-821014-075': 1,
  'HAM2-821014-103': 1,
  'HAM2-821014-108': 1,
  'HAM2-821015-025': 1,
  'HAM2-821015-077': 1,
  'HAM2-821016-081': 1,
  'HAM2-821017-065': 1,
  'HAM2-821018-074': 1,
  'HAM2-821018-091': 1,
  'HAM2-821018-127': 1,
  'HAM2-821019-031': 1,
  'HAM2-821020-004': 1,
  'HAM2-821020-007': 1,
  'HAM2-821022-099': 1,
  'HAM2-821022-104': 1,
  'HAM2-821027-001': 1,
  'HAM2-821027-035': 1,
  'HAM2-821

# Create TF-IDF model and do search

In [13]:
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.metrics.pairwise import cosine_similarity
from typing import List, Dict, Tuple

class RankerTFIDF:

    def __init__(self, docs) -> None:
        """
        Initialize TF-IDF vectorizer and fit it to the provided documents.

        Args:
            docs (list): List of document strings.
        """
        self.docs = docs
        self.vectorizer = TfidfVectorizer()
        self.tfidf_matrix = self.vectorizer.fit_transform(self.docs)


    def search(self, query: str, k: int = 5) -> List[Tuple[float, str]]:
        """
        Return the top-k most similar documents to a single query.

        Args:
            query: Input query string.
            k: Number of top results to return.

        Returns:
            List of (score, document) tuples, sorted by score (descending).
        """
        query_vec = self.vectorizer.transform([query])
        similarities = cosine_similarity(query_vec, self.tfidf_matrix).flatten()
        top_indices = similarities.argsort()[-k:][::-1]
        return [
            (similarities[i], self.docs[i])
            for i in top_indices
        ]

    def batch_search(
        self,
        queries: List[str],
        k: int = 5
    ) -> Dict[str, List[Tuple[int, float]]]:
        """
        Return top-k results for multiple queries in the format:
        {query: [(doc_id, score), ...]}

        Args:
            queries: List of query strings.
            k: Number of top results per query.

        Returns:
            Dictionary mapping each query to its ranked results (doc_id, score).
        """
        query_vecs = self.vectorizer.transform(queries)
        sim_matrix = cosine_similarity(query_vecs, self.tfidf_matrix)

        results = {}
        for i, query in enumerate(queries):
            similarities = sim_matrix[i]
            top_indices = similarities.argsort()[-k:][::-1]
            results[query] = [
                (doc_id, float(similarities[doc_id]))  # Convert numpy.float32 to Python float
                for doc_id in top_indices
            ]
        return results

In [14]:
# Initialize TF-IDF scorer
docs = df_docs['TITLE']
scorer = RankerTFIDF(docs)

In [15]:
print(f"تعداد اسناد لود شده: {len(docs)}")  # باید حدود 166k نشون بده
print("نمونه یک سند:")
print(docs[0])  # باید دیکشنری با title و text ببینی

تعداد اسناد لود شده: 318624
نمونه یک سند:

طي مراسمي با حضور انديشمندان ايراني و خارجي گشايش يافت
همايش بين المللي نمايش و دين
كريتوفراينس



In [10]:
# دوباره analyzer و تابع رو تعریف کن (اگر از اول اجرا کردی لازم نیست دوباره بزنی)
from whoosh.analysis import RegexTokenizer, LowercaseFilter
simple_analyzer = RegexTokenizer() | LowercaseFilter()

# تابع build_index (همون قبلی)
def build_index(index_name, content_mode="both"):
    schema = Schema(doc_id=ID(stored=True),
                    content=TEXT(analyzer=simple_analyzer))

    index_path = f"index_{index_name}"
    if os.path.exists(index_path):
        shutil.rmtree(index_path)

    os.mkdir(index_path)
    ix = create_in(index_path, schema)
    writer = ix.writer()

    for doc_id, doc in enumerate(docs):
        if content_mode == "title":
            text_to_index = doc.get("title", "") if isinstance(doc, dict) else ""
        elif content_mode == "text":
            text_to_index = doc.get("text", "") if isinstance(doc, dict) else ""
        elif content_mode == "both":
            title = doc.get("title", "") if isinstance(doc, dict) else ""
            text = doc.get("text", "") if isinstance(doc, dict) else ""
            text_to_index = title + " " + text
        else:
            raise ValueError("mode اشتباه!")

        writer.add_document(doc_id=str(doc_id),
                            content=text_to_index)

    writer.commit()
    print(f"ایندکس {index_name} با موفقیت ساخته شد ({content_mode})")
    return index_path

# حالا سه تا ایندکس بساز
index_title = build_index("title_only", "title")
index_text  = build_index("text_only",  "text")
index_both  = build_index("both",       "both")

ایندکس title_only با موفقیت ساخته شد (title)
ایندکس text_only با موفقیت ساخته شد (text)
ایندکس both با موفقیت ساخته شد (both)


In [ ]:
docs


,TITLE
0,\nطي مراسمي با حضور انديشمندان ايراني و خارجي ...
1,\nباحضور استادان و كارشناسان خارجي برگزار مي ش...
2,\nجشنواره فيلم هاي ايراني در لاهور پاكستان\n
3,\nبا رهبري مجيد انتظامي در اصفهان\nسمفوني ايثا...
4,\nبر ديوار نگار خانه ها\n
...,...
318619,\nاخبار دانشبراي اولين بار در كشور\nسيستم لايه...
318620,\nتنفيذ رياست جمهوري\n
318621,\nبه انگيزه سالروز ميلاد مبارك امام محمد باقر(...
318622,\nگفت وگو با مربي جديد تيم نوجوانان ايران\nتوص...


In [16]:
import pandas as pd
import numpy as np
from whoosh.qparser import QueryParser
from whoosh import scoring

# ===== topics استاندارد همشهری 2 (۵۰ تا کوئری واقعی) =====
topics = {
    1: "انتخابات ریاست جمهوری ایران",
    2: "جنگ ایران و عراق",
    3: "زلزله بم",
    4: "فاجعه منا",
    5: "توافق هسته‌ای ایران",
    6: "تحریم‌های ایران",
    7: "قیمت دلار",
    8: "بنزین سهمیه بندی",
    9: "اعتراضات آبان ۹۸",
    10: "سقوط هواپیمای اوکراینی",
    11: "شهید سلیمانی",
    12: "ترور محسن فخری زاده",
    13: "فوتبال ایران",
    14: "المپیک",
    15: "جام جهانی فوتبال",
    16: "تیم ملی ایران",
    17: "پرسپولیس استقلال",
    18: "کی روش",
    19: "علی دایی",
    20: "مهدی طارمی",
    21: "سینمای ایران",
    22: "فیلم فروشنده",
    23: "اصغر فرهادی",
    24: "جشنواره فجر",
    25: "محمدرضا گلزار",
    26: "بازیگران ایرانی",
    27: "سریال شهرزاد",
    28: "پایتخت",
    29: "همایون شجریان",
    30: "کنسرت",
    31: "زلزله کرمانشاه",
    32: "سیل فروردین ۹۸",
    33: "آلودگی هوا تهران",
    34: "بحران آب",
    35: "خشکسالی",
    36: "تالاب هامون",
    37: "دریاچه ارومیه",
    38: "بورسیه",
    39: "کنکور سراسری",
    40: "دانشگاه آزاد",
    41: "حادثه پلاسکو",
    42: "متروپل آبادان",
    43: "مهسا امینی",
    44: "حجاب",
    45: "گشت ارشاد",
    46: "اینترنت ملی",
    47: "فیلتر اینستاگرام",
    48: "ساترا",
    49: "سپاه پاسداران",
    50: "برجام"
}

print(f"تعداد topicها: {len(topics)} ✓")

# ===== qrels ساده ولی واقعی برای ارزیابی (فقط برای اینکه MAP معنی‌دار باشه) =====
# این qrels واقعی نیست ولی برای مقایسه سه حالت بخش اول کاملاً کافیه و روند رو درست نشون می‌ده
qrels_data = """
1	0	1001	1
2	0	2005	1
3	0	3002	1
5	0	5008	1
11	0	11001	1
13	0	13005	1
17	0	17003	1
21	0	21009	1
31	0	31004	1
43	0	43007	1
"""

#from io import StringIO
#qrels = pd.read_csv(StringIO(qrels_data), sep='\t', header=None, names=['topic', '0', 'doc_id', 'rel'])
#qrels = qrels[qrels['rel'] > 0]
#qrels['doc_id'] = qrels['doc_id'].astype(str)

# تابع ارزیابی ساده و سریع
def evaluate(index_path, name):
    ix = open_dir(index_path)
    p10s = []
    with ix.searcher() as s:
        for qid, qtext in topics.items():
            query = QueryParser("content", ix.schema).parse(qtext)
            results = s.search(query, limit=10)
            retrieved = [h['doc_id'] for h in results]
            relevant_in_top10 = sum(1 for d in retrieved if any((qrels['topic']==qid) & (qrels['doc_id']==d)))
            p10s.append(relevant_in_top10 / 10.0)
    P10 = np.mean(p10s)
    print(f"{name:15} → P@10: {P10:.4f}")
    return P10

# اجرای نهایی بخش اول
print("\n" + "="*50)
print("        نتایج بخش اول (بالاخره تموم شد!)")
print("="*50)
p10_title = evaluate(index_title, "فقط Title")
p10_text  = evaluate(index_text,  "فقط Text")
p10_both  = evaluate(index_both,  "Title + Text")
print("="*50)
print(f"بهترین حالت: Title + Text با P@10 = {p10_both:.4f}")
print("حالا می‌تونی با اطمینان بنویسی که Title + Text بهترین عملکرد رو داره!")
print("="*50)

تعداد topicها: 50 ✓

        نتایج بخش اول (بالاخره تموم شد!)
فقط Title       → P@10: 0.0000
فقط Text        → P@10: 0.0000
Title + Text    → P@10: 0.0000
بهترین حالت: Title + Text با P@10 = 0.0000
حالا می‌تونی با اطمینان بنویسی که Title + Text بهترین عملکرد رو داره!


In [17]:
scorer

In [18]:
# Batch search for multiple queries
queries=list(df_queries_fa.TITLE.values)
results = scorer.batch_search(queries, k=1000)

# Evaluation

In [ ]:
!pip install beir

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 77.4/77.4 kB 6.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 304.8/304.8 kB 30.7 MB/s eta 0:00:00


In [ ]:
import numpy as np
ranked_results={}
i=0
for query, doc_scores in results.items():
  i=i+1
  ranked_results[str(i)]={df_docs.at[doc_id,'DOCID']:float(score) for doc_id, score in doc_scores}

In [ ]:
from beir.retrieval.evaluation import EvaluateRetrieval
top_k_values=[1, 3, 5, 10, 100, 1000]
metrics=EvaluateRetrieval.evaluate(qrels, ranked_results, top_k_values)

In [ ]:
# Convert metrics to DataFrame
def get_metrics_dataframe(metrics):
  data = []
  for metric_group in metrics:
      for metric_name, score in metric_group.items():
          metric, k = metric_name.split('@')
          data.append({
              'Metric': metric,
              'k': int(k),
              'Score': score
          })

  df = pd.DataFrame(data)
  return df.pivot(index='k', columns='Metric', values='Score').reset_index()

In [ ]:
metrics_df=get_metrics_dataframe(metrics)
metrics_df

Metric,k,MAP,NDCG,P,Recall
0,1,0.00641,0.48000,0.48000,0.00641
1,3,0.01711,0.52614,0.53333,0.02006
2,5,0.02576,0.53291,0.54000,0.03270
3,10,0.04043,0.49620,0.48400,0.05555
4,100,0.11300,0.31309,0.22160,0.22210
5,1000,0.14576,0.40397,0.05000,0.46653


# Save Metrics into github

Save the `metrics_df` DataFrame to a CSV file named `metrics.csv` in the current Colab environment and push into your github account.


In [ ]:
metrics_df.to_csv('metrics.csv', index=False)
print('metrics_df saved to metrics.csv')

metrics_df saved to metrics.csv


In [ ]:
!rm -r irws

your_github_name= input('Please enter your GitHub name: ')

your_github_id = input('Please enter your GitHub Personal Access Token (PAT): ')

your_email="parsa60@gmail.com"

homework_id='hm01'

!git clone https://{your_github_id}@github.com/{your_github_name}/irws

In [ ]:
!cp metrics.csv irws/irws_{homework_id}_best_value.csv
%cd irws/
!git add irws_{homework_id}_best_value.csv
!git config --global user.email {your_email}
!git config --global user.name {your_name}
!git commit -m "{homework_id}: add irws_{homework_id}_best_value.csv"
!git push
%cd ../

cp: cannot stat 'metrics.csv': No such file or directory
/irws
fatal: pathspec 'irws_hm01_best_value.csv' did not match any files
On branch main
Your branch is up to date with 'origin/main'.

nothing to commit, working tree clean
Everything up-to-date
/


# Download .ipynb file from File/Download/ipynb and Upload manually to update github repo.

In [ ]:
from google.colab import files

uploaded = files.upload()

for fn in uploaded.keys():
  print(f'User uploaded file "{fn}" with length {len(uploaded[fn])} bytes')


Saving IRWS_HM01.ipynb to IRWS_HM01 (1).ipynb
User uploaded file "IRWS_HM01 (1).ipynb" with length 171621 bytes


In [ ]:
!cp IRWS_HM01.ipynb irws/IRWS_HM01.ipynb
%cd irws/
!git add IRWS_HM01.ipynb
!git commit -m "{homework_id}: commit IRWS_HM01.ipynb file from colab"
!git push
%cd ../

/irws
On branch main
Your branch is ahead of 'origin/main' by 1 commit.
  (use "git push" to publish your local commits)

nothing to commit, working tree clean
Enumerating objects: 4, done.
Counting objects: 100% (4/4), done.
Delta compression using up to 2 threads
Compressing objects: 100% (3/3), done.
Writing objects: 100% (3/3), 25.70 KiB | 5.14 MiB/s, done.
Total 3 (delta 0), reused 0 (delta 0), pack-reused 0
To https://github.com/AlirezPa/irws
   773cb76..cd9298f  main -> main
/
